# Data Processing Pipeline
Очистка и предобработка данных: удаление пропусков, кодирование, масштабирование.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## Загрузка и начальный анализ

In [ ]:
CSV_PATH = '../dataset/raw_house_dataset.csv'
df = pd.read_csv(CSV_PATH)
print(df.head())
print(df.info())
print(df.isnull().sum())

## Очистка данных

In [ ]:
df['Price'] = df['Price'].replace(r'[\$,]', '', regex=True).astype(float)
df['Location'] = df['Location'].replace({'Subrb': 'Suburb', '??': pd.NA})
df['Size_sqft'] = df['Size_sqft'].fillna(df['Size_sqft'].median())
df['Bedrooms'] = df['Bedrooms'].fillna(df['Bedrooms'].mode()[0])
df['Location'] = df['Location'].fillna(df['Location'].mode()[0])
df = df.drop_duplicates()
print(f'Размер после очистки: {df.shape}')

## IQR + кодирование + feature engineering

In [ ]:
def iqr_fun(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k*(q3-q1), q3 + k*(q3-q1)

lp, hp = iqr_fun(df['Price'])
ls, hs = iqr_fun(df['Size_sqft'])
df['Price'] = df['Price'].clip(lp, hp)
df['Size_sqft'] = df['Size_sqft'].clip(ls, hs)

df = pd.get_dummies(df, columns=['Location'], drop_first=False, dtype='int')

df['HouseAge'] = 2025 - df['YearBuilt']
df['Rooms_per_1000sqft'] = (df['Bedrooms']+df['Bathrooms'])/(df['Size_sqft']/1000)
df['Size_per_Bedroom'] = df['Size_sqft']/df['Bedrooms'].replace(0, np.nan)
df['Is_City'] = df['Location_City'].astype(int)
df['LogPrice'] = np.log1p(df['Price'])

## Масштабирование и сохранение

In [ ]:
dont_scale = {'Price', 'LogPrice'}
exclude = [c for c in df.columns if c.startswith('Location_')] + ['Is_City']
num_features = [c for c in df.select_dtypes(include=['int64','float64']).columns if c not in dont_scale and c not in exclude]

df[num_features] = StandardScaler().fit_transform(df[num_features])

print(df.head())
print(df.isnull().sum())

df.to_csv('../dataset/clean_house_dataset.csv', index=False)
print('Сохранено!')